# Lezione 9 — Assistente vocale conversazionale (STT + LLM + TTS)

## Obiettivi di apprendimento

- Capire la pipeline completa di un assistente vocale: voce → testo → risposta → voce.
- Riconoscere che un assistente vocale non è "un modello", ma tre pezzi indipendenti concatenati.
- Eseguire su Colab, 100% gratis, una pipeline STT (Whisper) + LLM (SmolLM2) + TTS (gTTS).
- Identificare il vuoto reale rimasto irrisolto nel materiale fonte di IBM (`text_to_speech`).

## Cosa costruiremo

Un mini assistente vocale end-to-end: generiamo un audio con una domanda,
lo trascriviamo, lo passiamo a un LLM, e convertiamo la sua risposta di nuovo in
audio. È esattamente la stessa architettura del progetto `chatapp-with-voice-and-openai-outline`
di IBM (`worker.py` + `server.py`), ma con pezzi gratuiti di Hugging Face al
posto di Watson e OpenAI.

## Appunti: la pipeline di un assistente vocale

Un assistente vocale conversazionale concatena tre funzioni indipendenti,
organizzate esattamente come nel `worker.py` reale del progetto di IBM:

1. **`speech_to_text(audio)`** — Speech-to-Text: converte l'audio in testo.
   Nel materiale originale usa IBM Watson STT; qui usiamo Whisper-tiny tramite
   Hugging Face.
2. **`llm_process_message(texto)`** — l'LLM pensa una risposta in testo.
   Nel materiale originale usa l'API di OpenAI; qui usiamo un modello locale
   piccolo (`SmolLM2-360M-Instruct`).
3. **`text_to_speech(texto)`** — Text-to-Speech: converte la risposta in
   audio. Nel materiale originale usa IBM Watson TTS; qui usiamo `gTTS`
   (Google Text-to-Speech, gratuito, richiede solo internet).

Un server Flask (`server.py`) collega questi tre pezzi con il browser,
ma in questo notebook li concateneremo direttamente in Python.

In [ ]:
# Installiamo le librerie necessarie (gTTS per il TTS, transformers/torch
# per STT e l'LLM). Su Colab, transformers e torch di solito sono già preinstallati.
!pip install -q gTTS transformers accelerate

### Passo 0: generiamo un audio "utente" per avere qualcosa da trascrivere

In un'app reale, questo audio arriverebbe dal microfono dell'utente registrato nel
browser. Qui, per poter eseguire l'intera pipeline in un notebook senza
microfono, generiamo noi stessi un audio di esempio con gTTS — così
simuliamo la voce di un utente che fa una domanda in inglese.

In [ ]:
from gtts import gTTS
from IPython.display import Audio

texto_usuario = "What is the capital of France?"
gTTS(text=texto_usuario, lang="en").save("audio_usuario.mp3")

# Riproduciamo l'audio generato per verificare che suoni bene
Audio("audio_usuario.mp3")

### Passo 1: Speech-to-Text (STT) — voce a testo

Usiamo lo stesso modello che già conosci dal capitolo su Whisper:
`openai/whisper-tiny.en` tramite `transformers.pipeline`. È piccolo, gira su
CPU, ed è la strada che usano i progetti reali di IBM per Colab gratuito.

In [ ]:
from transformers import pipeline

stt_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny.en",
)

def speech_to_text(audio_path):
    """Riceve il percorso di un file audio e restituisce il testo trascritto."""
    resultado = stt_pipe(audio_path)
    return resultado["text"].strip()

texto_transcrito = speech_to_text("audio_usuario.mp3")
print("Testo trascritto da Whisper:", texto_transcrito)

### Passo 2: l'LLM — testo a testo

Nel materiale originale di IBM, questo pezzo è `openai_process_message`,
che chiama l'API di OpenAI (a pagamento). Qui usiamo un modello piccolo e
gratuito di Hugging Face, `HuggingFaceTB/SmolLM2-360M-Instruct`, che hai già
usato nella lezione sui chatbot.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

def llm_process_message(user_message, system_prompt="Sei un assistente breve e cordiale."):
    """Invia il messaggio dell'utente all'LLM e restituisce la risposta in testo."""
    mensajes = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]
    entrada = tokenizer.apply_chat_template(
        mensajes, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )
    with torch.inference_mode():
        salida = llm_model.generate(entrada, max_new_tokens=80, do_sample=True, temperature=0.7)
    texto_generado = tokenizer.decode(salida[0][entrada.shape[1]:], skip_special_tokens=True)
    return texto_generado.strip()

respuesta = llm_process_message(texto_transcrito)
print("Risposta dell'LLM:", respuesta)

### Passo 3: Text-to-Speech (TTS) — testo a voce

Questo è, precisamente, il pezzo che nel `worker.py` reale di IBM è rimasto
con un `base_url = '...'` segnaposto, senza collegarsi davvero al servizio
a pagamento (Watson TTS). Qui lo completiamo con `gTTS`, che non richiede
nessuna chiave API.

In [ ]:
def text_to_speech(text, salida_path="respuesta.mp3", idioma="en"):
    """Converte il testo in un file audio e restituisce il percorso del file."""
    voz = gTTS(text=text, lang=idioma)
    voz.save(salida_path)
    return salida_path

ruta_audio_respuesta = text_to_speech(respuesta)
Audio(ruta_audio_respuesta)

## Riferimento: così si presenta con le API a pagamento del materiale ufficiale di IBM

Il codice qui sotto è quello reale di `worker.py` nel progetto
`chatapp-with-voice-and-openai-outline`. Non eseguirlo: avresti bisogno di
credenziali di IBM Watson e di OpenAI. Viene mostrato solo perché tu riconosca
che è esattamente la stessa architettura a tre funzioni, che dialoga con
servizi esterni a pagamento invece che con modelli locali.

```python
# worker.py (frammento reale, richiede credenziali)

def speech_to_text(audio_binary):
    api_url = base_url + '/speech-to-text/api/v1/recognize'
    params = {'model': 'en-US_Multimedia'}
    response = requests.post(api_url, params=params, data=audio_binary).json()
    text = 'null'
    while bool(response.get('results')):
        text = response.get('results').pop().get('alternatives').pop().get('transcript')
    return text

def openai_process_message(user_message):
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role": "user", "content": user_message}],
    )
    return response.choices[0].message.content

def text_to_speech(text, voice=""):
    # <-- nel materiale fonte, questa funzione aveva solo "base_url = '...'"
    #     come segnaposto, senza arrivare a collegarsi al servizio reale.
    api_url = base_url + '/text-to-speech/api/v1/synthesize?output=output_text.wav'
    if voice not in ("", "default"):
        api_url += "&voice=" + voice
    headers = {'Accept': 'audio/wav', 'Content-Type': 'application/json'}
    response = requests.post(api_url, headers=headers, json={'text': text})
    return response.content
```

**Nota importante**: nel repository reale di IBM, `text_to_speech` è rimasta
incompleta (con un segnaposto). Risolverai tu proprio questo stesso vuoto, con la
libreria gratuita, nel notebook di pratica di questa lezione.

## Riepilogo della lezione

- Un assistente vocale conversazionale = STT + LLM + TTS, tre pezzi
  indipendenti concatenati da un server (o, qui, da poche righe
  di Python).
- Ogni pezzo deve solo conoscere il formato di ingresso/uscita del pezzo precedente
  e di quello successivo: audio → testo → testo → audio.
- Abbiamo potuto eseguire l'intera pipeline gratis con Whisper-tiny (STT),
  SmolLM2 (LLM) e gTTS (TTS).
- Nel materiale reale di IBM, la funzione `text_to_speech` è rimasta con un
  segnaposto irrisolto — un vuoto genuino del laboratorio originale.

Continua con:
- Capitolo del manuale: `manual/13_asistente_voz_conversacional.md`
- Notebook di pratica: `notebooks_practica/clase09_practica_asistente_voz_conversacional.ipynb`